# Paper Feature Dump: A5alpha + CLIP-SENet

Slim GPU-only Kaggle kernel for retrieval-panel features. It does not train; it loads the fixed A5alpha TransReID checkpoint and the frozen CLIP-SENet checkpoint, extracts VeRi-776 query/gallery features, and writes NumPy arrays plus an index map under `/kaggle/working/features`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print("Python:", sys.version)
print("Installing notebook-only requirements...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "timm>=0.9.16",
        "open_clip_torch",
        "pretrainedmodels",
        "omegaconf>=2.3",
        "loguru",
    ],
    check=True,
)

REPO_DIR = Path("/kaggle/working/gp")
if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            "paper-tests",
            "https://github.com/MRKDaGods/gp.git",
            str(REPO_DIR),
        ],
        check=True,
    )
subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

In [ ]:
import gc
import json
import time
from pathlib import Path

import numpy as np
import torch

from src.serving.reid_loaders import (
    CLIPSENET_IMG_SIZE,
    TRANSREID_IMG_SIZE,
    build_09v_model,
    build_clipsenet_model,
    extract_09v_features_with_metadata,
    extract_clipsenet_features,
    parse_veri_split,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if DEVICE != "cuda":
    raise RuntimeError("This kernel requires Kaggle GPU; do not run feature extraction on CPU.")


def discover_veri_root() -> Path:
    candidates = []
    for split_dir in Path("/kaggle/input").rglob("image_query"):
        root = split_dir.parent
        if (root / "image_test").is_dir():
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError("Could not find VeRi-776 root with image_query/image_test under /kaggle/input")
    candidates.sort(key=lambda path: ("veri" not in str(path).lower(), str(path)))
    return candidates[0]


def find_required_file(filename: str) -> Path:
    matches = sorted(Path("/kaggle/input").rglob(filename), key=lambda path: str(path))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/input")
    return matches[0]


def discover_clipsenet_checkpoint() -> Path:
    candidates = []
    for name in ("clipsenet_v6_veri776_best.pth", "vehicle_clip_senet_veri776.pth", "best.pth", "best_mAP.pth"):
        for path in Path("/kaggle/input").rglob(name):
            score = 0
            text = str(path).lower()
            if "13-clip-senet-train" in text:
                score += 100
            if "clip" in text or "senet" in text:
                score += 20
            if name in {"clipsenet_v6_veri776_best.pth", "vehicle_clip_senet_veri776.pth", "best.pth"}:
                score += 5
            candidates.append((score, path))
    if not candidates:
        raise FileNotFoundError("Could not find CLIP-SENet checkpoint under /kaggle/input")
    candidates.sort(key=lambda row: (-row[0], str(row[1])))
    return candidates[0][1]

VERI_ROOT = discover_veri_root()
A5ALPHA_CKPT = find_required_file("a5alpha_checkpoint.pth")
CLIPSENET_CKPT = discover_clipsenet_checkpoint()
print("VERI_ROOT:", VERI_ROOT)
print("A5ALPHA_CKPT:", A5ALPHA_CKPT)
print("CLIPSENET_CKPT:", CLIPSENET_CKPT)

query_items, query_ids = parse_veri_split(VERI_ROOT / "image_query")
gallery_items, gallery_ids = parse_veri_split(VERI_ROOT / "image_test")
print(f"Query: {len(query_items):,} images, {query_ids} IDs")
print(f"Gallery: {len(gallery_items):,} images, {gallery_ids} IDs")

In [ ]:
OUT_DIR = Path("/kaggle/working/features")
(OUT_DIR / "stream1").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "stream2").mkdir(parents=True, exist_ok=True)


def save_feature(path: Path, array: np.ndarray) -> None:
    array = np.asarray(array, dtype=np.float16)
    np.save(path, array)
    print(f"saved {path}: shape={array.shape} dtype={array.dtype} size={path.stat().st_size / (1024 ** 2):.2f} MiB")


def index_records(items: list[dict], paths: list[str], pids: np.ndarray, camids: np.ndarray) -> list[dict]:
    records = []
    for row, (item, image_path, pid, camid) in enumerate(zip(items, paths, pids, camids)):
        records.append(
            {
                "row": int(row),
                "image_path": str(image_path),
                "vehicle_id": int(pid),
                "camera_id": int(camid),
                "split": "query" if "image_query" in str(image_path) else "gallery",
                "filename": Path(str(image_path)).name,
                "relative_path": f"{Path(str(image_path)).parent.name}/{Path(str(image_path)).name}",
            }
        )
    return records

started = time.time()
print("Loading Stream 1 A5alpha TransReID...")
stream1_model = build_09v_model(A5ALPHA_CKPT, DEVICE)
q_s1, q_pid_s1, q_cam_s1, q_paths = extract_09v_features_with_metadata(
    stream1_model,
    query_items,
    DEVICE,
    batch_size=64,
    stream="concat_patch_flip",
)
g_s1, g_pid_s1, g_cam_s1, g_paths = extract_09v_features_with_metadata(
    stream1_model,
    gallery_items,
    DEVICE,
    batch_size=64,
    stream="concat_patch_flip",
)
print("Stream 1:", q_s1.shape, g_s1.shape)
del stream1_model
gc.collect()
torch.cuda.empty_cache()

print("Loading Stream 2 CLIP-SENet...")
stream2_model = build_clipsenet_model(CLIPSENET_CKPT, DEVICE)
q_s2, q_pid_s2, q_cam_s2, q_paths_s2 = extract_clipsenet_features(
    stream2_model,
    query_items,
    CLIPSENET_IMG_SIZE,
    batch_size=32,
    device=DEVICE,
)
g_s2, g_pid_s2, g_cam_s2, g_paths_s2 = extract_clipsenet_features(
    stream2_model,
    gallery_items,
    CLIPSENET_IMG_SIZE,
    batch_size=32,
    device=DEVICE,
)
print("Stream 2:", q_s2.shape, g_s2.shape)
assert np.array_equal(q_pid_s1, q_pid_s2)
assert np.array_equal(g_pid_s1, g_pid_s2)
assert np.array_equal(q_cam_s1, q_cam_s2)
assert np.array_equal(g_cam_s1, g_cam_s2)

save_feature(OUT_DIR / "stream1" / "query.npy", q_s1)
save_feature(OUT_DIR / "stream1" / "gallery.npy", g_s1)
save_feature(OUT_DIR / "stream2" / "query.npy", q_s2)
save_feature(OUT_DIR / "stream2" / "gallery.npy", g_s2)

index_map = {
    "exp_id": "A5alpha",
    "feature_stage": "pre_aqe_pre_rerank",
    "source_kernel": "gumfreddy/paper-features-a5alpha",
    "veri_root": str(VERI_ROOT),
    "checkpoints": {
        "stream1": str(A5ALPHA_CKPT),
        "stream2": str(CLIPSENET_CKPT),
    },
    "stream1": {"dim": int(q_s1.shape[1]), "tta": "concat_patch_flip", "image_size": list(TRANSREID_IMG_SIZE)},
    "stream2": {"dim": int(q_s2.shape[1]), "tta": "none", "image_size": list(CLIPSENET_IMG_SIZE)},
    "query": index_records(query_items, q_paths, q_pid_s1, q_cam_s1),
    "gallery": index_records(gallery_items, g_paths, g_pid_s1, g_cam_s1),
}
(OUT_DIR / "index_map.json").write_text(json.dumps(index_map, indent=2), encoding="utf-8")
print("saved", OUT_DIR / "index_map.json")
print(json.dumps({
    "elapsed_minutes": round((time.time() - started) / 60.0, 2),
    "stream1_query": list(q_s1.shape),
    "stream1_gallery": list(g_s1.shape),
    "stream2_query": list(q_s2.shape),
    "stream2_gallery": list(g_s2.shape),
}, indent=2))
print("Final feature files:")
for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file():
        print(path, path.stat().st_size)